# Data Cleaning — E-commerce Orders
**Oasis Infobyte | Data Analytics | Level 1 — Task 3**

## Objective
Clean a deliberately messy e-commerce orders dataset and document every data-quality decision. The included dataset is **synthetic practice data**, not real customer records.

## Project files
- `data/messy_ecommerce_orders.csv` — intentionally messy input.
- `outputs/cleaned_ecommerce_orders.csv` — cleaned dataset.
- `outputs/initial_data_quality_report.csv` — initial quality report.
- `outputs/before_after_quality_summary.csv` — before/after summary.

## Cleaning plan
1. Inspect rows, columns, types, nulls and duplicates.
2. Standardise text, IDs, gender, city and order status.
3. Convert dates and numeric fields safely.
4. Handle missing/invalid values using documented rules.
5. Detect extreme unit-price values with an IQR check and cap/remove invalid extreme values from analysis.
6. Export a clean CSV.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("data/messy_ecommerce_orders.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

raw = pd.read_csv(DATA_PATH)
df = raw.copy()
print("Dataset loaded:", raw.shape)
display(raw.head())


## 2. Initial data quality report

In [ ]:
quality_report = pd.DataFrame({
    "missing_values": raw.isna().sum(),
    "data_type": raw.dtypes.astype(str),
    "unique_values": raw.nunique(dropna=True)
})
print("Rows and columns:", raw.shape)
print("Duplicate rows:", raw.duplicated().sum())
display(quality_report)


## 3. Cleaning decisions

- **Duplicates:** exact duplicate records are removed because they represent repeated rows.
- **Text formatting:** leading/trailing spaces are stripped; IDs are uppercase; names and city labels use title case.
- **Gender:** values such as `M`, `male`, `F`, and `female` are mapped to `Male` and `Female`. Unrecognised values become `Unknown`.
- **Dates:** parse to datetime; invalid dates become missing. Rows without a valid order date are removed because time-based order analysis would be unreliable.
- **Quantity:** convert to numeric; zero/negative or missing quantities are replaced with the median valid quantity.
- **Unit price:** remove commas before numeric conversion; invalid/missing/non-positive prices are imputed with the median valid price. Extremely high prices above ₹5,000 are treated as anomalies and imputed rather than allowed to distort totals.
- **Other categorical missing values:** fill with `Unknown` so the row is retained without inventing a category.
- **Outliers:** inspect unit-price IQR bounds; only clearly implausible extreme values are replaced according to the documented ₹5,000 business-rule threshold.


## 4. Remove duplicates and standardise text

In [ ]:
rows_before = len(df)
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

df["order_id"] = df["order_id"].str.upper()
df["customer_id"] = df["customer_id"].str.upper()
df["customer_name"] = df["customer_name"].str.title()
df["city"] = df["city"].str.title()
df["status"] = df["status"].str.lower().str.title()

def standardise_gender(row):
    value = str(row["gender"]).strip().lower()
    code = str(row["gender_code"]).strip().lower()
    if value in ["m", "male"] or code in ["m", "male"]:
        return "Male"
    if value in ["f", "female"] or code in ["f", "female"]:
        return "Female"
    return "Unknown"

df["gender"] = df.apply(standardise_gender, axis=1)
df = df.drop(columns=["gender_code"])
print("Duplicate rows removed:", duplicates_removed)


## 5. Correct data types and handle invalid values

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(
    df["unit_price"].astype("string").str.replace(",", "", regex=False),
    errors="coerce"
)

# Treat zero/negative quantities and prices as invalid.
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] <= 0, "unit_price"] = np.nan

# Inspect IQR bounds for unit price.
q1, q3 = df["unit_price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("Unit price IQR bounds:", round(lower_bound, 2), "to", round(upper_bound, 2))
print("Unit price values above ₹5,000:", int((df["unit_price"] > 5000).sum()))

# Project-specific plausibility rule: prices above ₹5,000 are treated as entry errors.
df.loc[df["unit_price"] > 5000, "unit_price"] = np.nan

# Median imputation is robust to skew and outliers.
df["quantity"] = df["quantity"].fillna(df["quantity"].median()).round().astype("int64")
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median()).round(2)

# Fill non-critical missing categories with an explicit label.
for col in ["customer_name", "customer_id", "gender", "city", "status"]:
    df[col] = df[col].fillna("Unknown")

# Invalid dates cannot support time-based reporting, so remove those rows.
invalid_dates = int(df["order_date"].isna().sum())
df = df.dropna(subset=["order_date"]).copy()
df["revenue"] = (df["quantity"] * df["unit_price"]).round(2)

print("Rows removed due to invalid dates:", invalid_dates)
print("Cleaned shape:", df.shape)
display(df.head())


## 6. Before vs. after summary

In [ ]:
before_summary = {
    "Rows": len(raw),
    "Duplicate rows": int(raw.duplicated().sum()),
    "Missing values": int(raw.isna().sum().sum())
}
after_summary = {
    "Rows": len(df),
    "Duplicate rows": int(df.duplicated().sum()),
    "Missing values": int(df.isna().sum().sum())
}
comparison = pd.DataFrame({"Metric": before_summary.keys(),
                           "Before": before_summary.values(),
                           "After": after_summary.values()})
display(comparison)


## 7. Save analysis-ready dataset

In [ ]:
clean_path = OUTPUT_DIR / "cleaned_ecommerce_orders.csv"
df.to_csv(clean_path, index=False)
print("Cleaned dataset saved to:", clean_path)
print("Final data types:")
display(df.dtypes.to_frame("dtype"))


## 8. Final observations and conclusion

The cleaning workflow removes exact duplicates, standardises inconsistent labels, converts dates and numeric fields, handles missing values with transparent rules, and prevents an extreme unit-price entry from distorting revenue. The resulting CSV is suitable for further exploratory analysis and dashboard development.

### Limitations
- This is a small synthetic demonstration dataset created for learning.
- Median imputation and the ₹5,000 price threshold are project assumptions; a real business should confirm them with domain experts.
- `Unknown` values are retained where guessing would be inappropriate.

### Recommendations
1. Enforce date, quantity and price validation at the data-entry stage.
2. Use controlled dropdown values for gender, city and order status.
3. Run a scheduled data-quality report before every monthly sales dashboard refresh.
